I will now continue with Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv, having Web Attacks 
the CIC dataset is actually flow based
Because a Web Attack (like XSS or Brute Force) just looks like normal HTTP web traffic on the surface, this is one of the hardest attacks for flow-based models to detect. The models have to rely entirely on tiny variations in timing and payload size.
So it can show quite a challenge to be working through these

i will still use the smote and scaling strategy, considering that there is imbalance here as well

In [13]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

In [14]:
print(" Loading the Web Attacks Dataset")
file_path = "MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

print("\n Traffic Distribution (Before Cleaning)")
print(df['Label'].value_counts())

 Loading the Web Attacks Dataset

 Traffic Distribution (Before Cleaning)
Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64


In [15]:
print("\n Cleaning and Encoding Data")
df_clean = df.dropna()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

#did combine the attacks here
df_clean['Label'] = df_clean['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print("\n Traffic Distribution (Before Cleaning)")
print(df_clean['Label'].value_counts())


 Cleaning and Encoding Data

 Traffic Distribution (Before Cleaning)
Label
0    168051
1      2180
Name: count, dtype: int64


In [16]:
print("\n Splitting Data")
X = df_clean.drop('Label', axis=1)
y = df_clean['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set Label Count BEFORE SMOTE:\n{y_train.value_counts()}")


 Splitting Data
Training Set Label Count BEFORE SMOTE:
Label
0    134448
1      1736
Name: count, dtype: int64


In [17]:
print("\n Applying SMOTE to balance the Training Data.")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"Training Set Label Count AFTER SMOTE:\n{y_train_smote.value_counts()}")


 Applying SMOTE to balance the Training Data.
Training Set Label Count AFTER SMOTE:
Label
0    134448
1    134448
Name: count, dtype: int64


In [18]:
print("\n Scaling the Data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote) 
X_test_scaled = scaler.transform(X_test)


 Scaling the Data...


In [19]:
print("\n Multi-Model Cycling")

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(50,), max_iter=100, random_state=42)
}

results = []


 Multi-Model Cycling


In [20]:
for name, model in models.items():
    print(f"  -> Training {name}...")
    start_time = time.time()
    
    model.fit(X_train_scaled, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    train_time = time.time() - start_time
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred) 
    
    results.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Recall (Attack Catch Rate)": round(recall * 100, 4),
        "F1-Score": round(f1, 4),
        "Training Time (s)": round(train_time, 2)
    })

  -> Training Decision Tree...
  -> Training Random Forest...
  -> Training XGBoost...
  -> Training LightGBM...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  -> Training Logistic Regression...
  -> Training MLP (Neural Net)...


c:\Users\Magda\Desktop\AnomalyDetection\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [21]:
print("\n Final Web Attack Model Leaderboard")
results_df = pd.DataFrame(results).sort_values(by="Recall (Attack Catch Rate)", ascending=False)
display(results_df)


 Final Web Attack Model Leaderboard


,Model,Accuracy (%),Recall (Attack Catch Rate),F1-Score,Training Time (s)
2,XGBoost,99.9971,100.0000,0.9989,5.21
0,Decision Tree,99.9794,99.7748,0.9922,11.01
3,LightGBM,99.9941,99.7748,0.9977,5.54
5,MLP (Neural Net),99.7797,99.3243,0.9216,316.13
1,Random Forest,99.9824,98.8739,0.9932,9.05
4,Logistic Regression,98.5285,98.4234,0.6356,23.42


First i will talk about mlp as it gave a warning. in my code earlier i wrote for it to have a max of 100 iterations, and now it has hit that number and stopped itself. Because the normal traffic was too similar to the attacks, mlp kept trying to change it s mathematical formula over and over again (to that 100th time) and then it gave up and used the best score it had This is why in this it took less compared to when i did the infiltration attacks 

For the rest we have the best one which is XGboost, which hit 100 recall which means that it caught every single attack, the f1 score is almost of 1, so the amount of false positives has been really low. Plus the time of execution was short of just 5 seconds which was perfect for what we gave it 

The worst one again is Logistic Regression that struggled just like it did for infiltration attacks with the synthetic smote data. Even though it has a decent recall score, the f1 shows it had a lot of false-positives. This is important proof that simple linear models are not built for imbalanced data.

Random forest also dropped in recall, but the f1 is still good.